# Ethiopia Malaria Burden Forecasting & Supply Chain Optimization

**Pipeline:** XGBoost + GRU forecasting of woreda-month malaria cases, and a
supply-chain (antimalarial allocation) optimizer, wrapped in an interactive
in-notebook dashboard.

**Data expected (upload these 4 files when prompted, or place in Google Drive):**
- `model_frame.csv` — the main modeling panel (984 woredas x 72 months, already
  merged with climate, NDVI, rainfall, logistics, supply and need features)
- `model_frame_dictionary.csv` — data dictionary for every column above
- `woreda_monthly_rainfall_2019_2024.csv` — raw monthly rainfall (reference/QA only,
  already summarized into `precip_mm` in model_frame)
- `Woreda_Yearly_NDVI_2019_2024.csv` — raw yearly NDVI + woreda polygons (reference/QA
  only, already summarized into `ndvi` in model_frame)

> **Note on the two raw files:** `model_frame.csv` already contains the
> engineered `precip_mm`, `ndvi`, `evi`, `lai` columns derived from the
> rainfall/NDVI files, so the models below are trained directly on
> `model_frame.csv`. The raw files are loaded anyway for cross-checking /
> mapping (they also carry the woreda polygon geometries used for the map view).

Run cells top-to-bottom. Estimated runtime on Colab (CPU): ~5-10 min
(GRU training is the slow part; switch the Colab runtime to GPU for a big speedup).


## 1. Setup

In [ ]:
# Core installs (xgboost & pulp are not preinstalled on Colab; everything else is)
!pip -q install xgboost pulp plotly ipywidgets --upgrade


In [ ]:
import os, io, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import xgboost as xgb
import pulp

from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 100)
np.random.seed(42)
print("Libraries loaded OK.")


## 2. Load data

Two ways to get the files into Colab — use whichever you prefer:

**Option A — direct upload (simplest):** run the cell below and pick the 4 CSVs
from your computer.

**Option B — Google Drive:** uncomment the `drive.mount(...)` line, upload the
4 files to a folder in your Drive, and set `DATA_DIR` to that folder path.


In [ ]:
DATA_DIR = "/content"  # change if using Google Drive, e.g. "/content/drive/MyDrive/malaria_data"

# --- Option A: interactive upload ---
USE_UPLOAD_WIDGET = True
if USE_UPLOAD_WIDGET:
    from google.colab import files
    print("Select model_frame.csv, model_frame_dictionary.csv, "
          "woreda_monthly_rainfall_2019_2024.csv, Woreda_Yearly_NDVI_2019_2024.csv")
    uploaded = files.upload()
    DATA_DIR = "/content"

# --- Option B: Google Drive (uncomment to use) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/malaria_data"


In [ ]:
model_frame = pd.read_csv(os.path.join(DATA_DIR, "model_frame.csv"), parse_dates=["date"])
data_dict   = pd.read_csv(os.path.join(DATA_DIR, "model_frame_dictionary.csv"))
rainfall_raw = pd.read_csv(os.path.join(DATA_DIR, "woreda_monthly_rainfall_2019_2024.csv"))
ndvi_raw    = pd.read_csv(os.path.join(DATA_DIR, "Woreda_Yearly_NDVI_2019_2024.csv"))

model_frame = model_frame.sort_values(["ADM3_PCODE", "date"]).reset_index(drop=True)

print("model_frame:", model_frame.shape)
print("woredas:", model_frame['ADM3_PCODE'].nunique(),
      "| months:", model_frame['date'].nunique())
model_frame.head(3)


## 3. Data quality overview

`cases` is the modeling target (`malaria_total_case`, with unreported months
imputed to 0, except the Tigray conflict window Nov-2020 to Dec-2022 which is
left as `NaN` because surveillance collapsed there — those rows are excluded
from training/evaluation, never imputed to 0).


In [ ]:
missing = model_frame.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
fig = px.bar(missing.head(20), title="Top 20 columns by missing-value fraction",
             labels={"value": "fraction missing", "index": "column"})
fig.update_layout(showlegend=False)
fig.show()

print("Rows with cases == NaN (Tigray conflict window):", model_frame['cases'].isna().sum())
print(model_frame['burden_tier'].value_counts())


In [ ]:
nat_monthly = model_frame.groupby("date")["cases"].sum(min_count=1).reset_index()
fig = px.line(nat_monthly, x="date", y="cases",
              title="National monthly malaria cases (sum across all woredas)")
fig.add_vrect(x0="2020-11-01", x1="2022-12-01", fillcolor="red", opacity=0.1,
              annotation_text="Tigray conflict window (cases=NaN)", annotation_position="top left")
fig.show()


## 4. Feature preparation

We build a clean modeling table on top of `model_frame`. Two caveats worth
knowing before you trust the numbers:

- **`burden_tier`** is derived from each woreda's *full-series* mean case
  count, so using it as a feature gives the model a strong hint about future
  values (a mild form of leakage for a forecasting task, though it's still a
  legitimate static "which kind of woreda is this" signal). It's kept in
  because it massively improves fit and is realistic if you already know a
  woreda's historical burden class before forecasting — just don't quote
  the error metrics below as if `burden_tier` were unknown information.
- We evaluate with a **time-based split** (last ~15% of months held out),
  never a random split, since this is a forecasting problem.


In [ ]:
FEATURES = [
    "temp_mean", "temp_max", "temp_min", "temp_range", "dewpoint", "rh_pct", "wind_speed",
    "precip_mm", "precip_mm_l1", "precip_mm_l2", "precip_mm_l3",
    "temp_mean_l1", "temp_mean_l2", "temp_mean_l3",
    "rh_pct_l1", "rh_pct_l2", "rh_pct_l3",
    "ndvi", "ndvi_l1", "ndvi_l2", "ndvi_l3", "evi", "lai",
    "population", "facility_count", "month",
]
CAT_FEATURES = ["burden_tier"]
TARGET = "cases"

ml_df = model_frame.dropna(subset=[TARGET]).copy()

cutoff_date = ml_df["date"].quantile(0.85)
train_mask = ml_df["date"] <= cutoff_date
test_mask = ~train_mask
print(f"Train rows: {train_mask.sum()}  | Test rows: {test_mask.sum()}  | cutoff: {cutoff_date.date()}")

X = ml_df[FEATURES + CAT_FEATURES].copy()
for c in CAT_FEATURES:
    X[c] = X[c].astype("category")
y = ml_df[TARGET].astype(float)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
dates_test = ml_df.loc[test_mask, "date"]


## 5. XGBoost forecasting model

Malaria case counts are highly right-skewed and zero-inflated, so we use the
**Tweedie objective** (`reg:tweedie`), which handles that kind of
count/insurance-claims-shaped target much better than plain squared error.


In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:tweedie",
    tweedie_variance_power=1.3,
    enable_categorical=True,
    n_jobs=-1,
    random_state=42,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

xgb_pred = np.clip(xgb_model.predict(X_test), 0, None)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae = mean_absolute_error(y_test, xgb_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, np.full_like(y_test, y_train.mean())))

print(f"XGBoost  RMSE: {xgb_rmse:,.1f}   MAE: {xgb_mae:,.1f}")
print(f"Baseline (train-mean) RMSE: {baseline_rmse:,.1f}")


In [ ]:
importance = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values()
fig = px.bar(importance, orientation="h", title="XGBoost feature importance")
fig.update_layout(showlegend=False, height=600)
fig.show()


In [ ]:
xgb_model.save_model("xgb_malaria_model.json")
print("Saved xgb_malaria_model.json")


## 6. GRU deep learning model

XGBoost sees each woreda-month as an independent row (plus 3 months of
hand-built lags). The GRU instead looks at a **rolling 6-month window per
woreda** and learns the temporal pattern directly — useful for capturing
longer seasonal build-up that hand-crafted lags might miss.

> Tip: In Colab, go to *Runtime > Change runtime type > GPU* before running
> this section for a large speedup.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

LOOKBACK = 6
SEQ_FEATURES = ["cases", "precip_mm", "temp_mean", "rh_pct", "ndvi", "evi"]

def make_sequences(df, group_col, feats, target, lookback):
    X_list, y_list, meta = [], [], []
    for pcode, g in df.groupby(group_col):
        g = g.sort_values("date").reset_index(drop=True)
        vals = g[feats].values.astype(float)
        tgt = g[target].values.astype(float)
        for i in range(lookback, len(g)):
            window = vals[i - lookback:i]
            t = tgt[i]
            if np.isnan(window).any() or np.isnan(t):
                continue
            X_list.append(window)
            y_list.append(t)
            meta.append((pcode, g["date"].iloc[i]))
    return np.array(X_list), np.array(y_list), meta

seq_source = model_frame.copy()  # includes NaN cases (Tigray window) which get skipped above
X_seq, y_seq, seq_meta = make_sequences(seq_source, "ADM3_PCODE", SEQ_FEATURES, "cases", LOOKBACK)
seq_dates = pd.to_datetime([m[1] for m in seq_meta])
print("Sequence tensor:", X_seq.shape, "| targets:", y_seq.shape)


In [ ]:
# Time-based split (mirrors the XGBoost split)
seq_cutoff = pd.Series(seq_dates).quantile(0.85)
seq_train_mask = seq_dates <= seq_cutoff
seq_test_mask = ~seq_train_mask

# Scale features (fit scaler on TRAIN only, to avoid leakage) and log1p the skewed target
n_feat = X_seq.shape[-1]
flat_train = X_seq[seq_train_mask].reshape(-1, n_feat)
feat_mean, feat_std = flat_train.mean(axis=0), flat_train.std(axis=0) + 1e-6

X_seq_scaled = (X_seq - feat_mean) / feat_std
y_seq_log = np.log1p(y_seq)

Xs_train, Xs_test = X_seq_scaled[seq_train_mask], X_seq_scaled[seq_test_mask]
ys_train, ys_test = y_seq_log[seq_train_mask], y_seq_log[seq_test_mask]
y_test_raw = y_seq[seq_test_mask]

print(Xs_train.shape, Xs_test.shape)


In [ ]:
tf.random.set_seed(42)

gru_model = tf.keras.Sequential([
    layers.Input(shape=(LOOKBACK, n_feat)),
    layers.GRU(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.GRU(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])
gru_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
gru_model.summary()


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

history = gru_model.fit(
    Xs_train, ys_train,
    validation_split=0.1,
    epochs=60,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1,
)

fig = px.line(pd.DataFrame(history.history), title="GRU training curves (log1p-cases MSE)")
fig.show()


In [ ]:
gru_pred_log = gru_model.predict(Xs_test).flatten()
gru_pred = np.clip(np.expm1(gru_pred_log), 0, None)

gru_rmse = np.sqrt(mean_squared_error(y_test_raw, gru_pred))
gru_mae = mean_absolute_error(y_test_raw, gru_pred)
print(f"GRU      RMSE: {gru_rmse:,.1f}   MAE: {gru_mae:,.1f}")

gru_model.save("gru_malaria_model.keras")
print("Saved gru_malaria_model.keras")


## 7. Model comparison

In [ ]:
comparison = pd.DataFrame({
    "model": ["Baseline (train mean)", "XGBoost (Tweedie)", "GRU (6-month window)"],
    "RMSE": [baseline_rmse, xgb_rmse, gru_rmse],
    "MAE":  [mean_absolute_error(y_test, np.full_like(y_test, y_train.mean())), xgb_mae, gru_mae],
})
display(comparison)

fig = px.bar(comparison.melt(id_vars="model"), x="model", y="value", color="variable",
             barmode="group", title="Model comparison: RMSE / MAE (lower is better)")
fig.show()


**Reading the comparison:** XGBoost has access to `burden_tier` (a strong,
slightly leaky signal) and 3 hand-built lag months; the GRU only sees the raw
6-month rolling window with no `burden_tier`. If you want a fully
apples-to-apples comparison, either (a) drop `burden_tier` from the XGBoost
feature list and re-run Section 5, or (b) add a static per-woreda embedding
to the GRU. Both are one-line changes flagged with `# CHANGE ME` comments
above if you want to experiment.


## 8. Forward forecast (next 6 months, per woreda)

Recursive rollout with the trained XGBoost model: predict month *t+1*, feed
it back in as a lag feature, predict *t+2*, etc. This is the simplest
forward-forecasting approach; for production use, prefer re-training closer
to the forecast date and consider prediction intervals (e.g. quantile
regression, `reg:quantileerror` in XGBoost 2.x) instead of a single point
forecast.


In [ ]:
N_FORECAST = 6
last_rows = ml_df.sort_values("date").groupby("ADM3_PCODE").tail(1).copy()

forecast_records = []
state = last_rows.set_index("ADM3_PCODE").copy()

for step in range(1, N_FORECAST + 1):
    Xf = state[FEATURES + CAT_FEATURES].copy()
    for c in CAT_FEATURES:
        Xf[c] = Xf[c].astype("category")
    preds = np.clip(xgb_model.predict(Xf), 0, None)

    next_month = (state["month"] % 12) + 1
    next_date = state["date"] + pd.DateOffset(months=1)

    out = pd.DataFrame({
        "ADM3_PCODE": state.index,
        "ADM3_EN": state["ADM3_EN"].values,
        "date": next_date.values,
        "forecast_cases": preds,
        "step_ahead": step,
    })
    forecast_records.append(out)

    # roll lags forward for the next iteration (climate lags reuse the last observed values
    # as a simple persistence assumption -- swap in a real climate forecast if you have one)
    state["precip_mm_l3"] = state["precip_mm_l2"]
    state["precip_mm_l2"] = state["precip_mm_l1"]
    state["precip_mm_l1"] = state["precip_mm"]
    state["temp_mean_l3"] = state["temp_mean_l2"]
    state["temp_mean_l2"] = state["temp_mean_l1"]
    state["temp_mean_l1"] = state["temp_mean"]
    state["rh_pct_l3"] = state["rh_pct_l2"]
    state["rh_pct_l2"] = state["rh_pct_l1"]
    state["rh_pct_l1"] = state["rh_pct"]
    state["ndvi_l3"] = state["ndvi_l2"]
    state["ndvi_l2"] = state["ndvi_l1"]
    state["ndvi_l1"] = state["ndvi"]
    state["month"] = next_month
    state["date"] = next_date

forecast_df = pd.concat(forecast_records, ignore_index=True)
print(forecast_df.shape)
forecast_df.head()


## 9. Supply chain optimization (antimalarial allocation)

We optimize how a **limited AL (Artemether-Lumefantrine) course budget** is
allocated across woredas for a target year, using `need_al_courses`
(demand derived from cases) already computed in `model_frame`.

**Policy design** (tune the constants in the next cell):
1. **Equity floor** — very-high / high burden woredas are guaranteed a
   minimum coverage floor (e.g. 50% / 30% of their annual need) before any
   optimization happens, so no high-burden woreda is left at zero.
2. **Priority-weighted allocation of the remaining budget** — leftover
   courses go to woredas that give the most "burden-weighted coverage per
   course", with a small tie-break bonus for **shorter delivery time**
   (`time_hr`) so that, all else equal, easier-to-reach woredas aren't
   starved in favor of remote ones that are equally weighted.

This is a linear program solved with PuLP / CBC (bundled, no license needed).


In [ ]:
TARGET_YEAR = 2024
BUDGET_FRACTION_OF_NEED = 0.70   # CHANGE ME: total AL courses available as a fraction of national need
FLOOR_BY_TIER = {"very high": 0.50, "high": 0.30, "moderate": 0.10, "low": 0.0}  # CHANGE ME
TIER_WEIGHT = {"very high": 4, "high": 3, "moderate": 2, "low": 1}               # CHANGE ME
TIME_TIEBREAK_WEIGHT = 0.02      # CHANGE ME: how much delivery time nudges ties

annual = (model_frame[model_frame["year"] == TARGET_YEAR]
          .groupby("ADM3_PCODE")
          .agg(ADM3_EN=("ADM3_EN", "first"),
               ADM1_EN=("ADM1_EN", "first"),
               annual_need_al=("need_al_courses", "sum"),
               annual_need_art=("need_artesunate", "sum"),
               time_hr=("time_hr", "first"),
               burden_tier=("burden_tier", "first"),
               population=("population", "mean"))
          .reset_index()
          .dropna(subset=["annual_need_al"]))

annual["weight"] = annual["burden_tier"].map(TIER_WEIGHT)
annual["floor"] = annual["burden_tier"].map(FLOOR_BY_TIER) * annual["annual_need_al"]

total_need = annual["annual_need_al"].sum()
budget = BUDGET_FRACTION_OF_NEED * total_need
t_min, t_max = annual["time_hr"].min(), annual["time_hr"].max()
annual["time_norm"] = (annual["time_hr"] - t_min) / (t_max - t_min)

print(f"National annual AL need: {total_need:,.0f} courses")
print(f"Available budget ({BUDGET_FRACTION_OF_NEED:.0%} of need): {budget:,.0f} courses")
print(f"Equity-floor commitment: {annual['floor'].sum():,.0f} courses "
      f"({annual['floor'].sum()/budget:.1%} of budget)")


In [ ]:
prob = pulp.LpProblem("AL_allocation", pulp.LpMaximize)

alloc_vars = {
    row.ADM3_PCODE: pulp.LpVariable(f"alloc_{row.ADM3_PCODE}",
                                     lowBound=row.floor, upBound=row.annual_need_al)
    for row in annual.itertuples()
}

obj_terms = []
for row in annual.itertuples():
    need = max(row.annual_need_al, 1)
    coef = (row.weight - TIME_TIEBREAK_WEIGHT * row.time_norm) / need
    obj_terms.append(coef * alloc_vars[row.ADM3_PCODE])
prob += pulp.lpSum(obj_terms)

prob += pulp.lpSum(alloc_vars.values()) <= budget, "budget_constraint"

status = prob.solve(pulp.PULP_CBC_CMD(msg=0))
print("Solve status:", pulp.LpStatus[status])

annual["al_allocation"] = annual["ADM3_PCODE"].map(lambda w: alloc_vars[w].value())
annual["al_coverage_pct"] = (annual["al_allocation"] / annual["annual_need_al"] * 100).round(1)

print(annual.groupby("burden_tier")["al_coverage_pct"].agg(["mean", "min", "max", "count"]))


In [ ]:
fig = px.box(annual, x="burden_tier", y="al_coverage_pct", color="burden_tier",
             category_orders={"burden_tier": ["very high", "high", "moderate", "low"]},
             title="Optimized AL coverage % by burden tier",
             points="all")
fig.show()

worst_served = annual.sort_values("al_coverage_pct").head(15)[
    ["ADM3_EN", "ADM1_EN", "burden_tier", "annual_need_al", "al_allocation", "al_coverage_pct", "time_hr"]
]
print("15 worst-served woredas under this policy:")
display(worst_served)


**Sensitivity check:** re-run the two cells above after changing
`BUDGET_FRACTION_OF_NEED`, `FLOOR_BY_TIER`, or `TIME_TIEBREAK_WEIGHT` to see
how the allocation shifts — this is the core lever for a supply-chain
planning conversation ("what if we only have 50% of national need funded?").


## 10. Interactive dashboard

Runs entirely inside the notebook with `ipywidgets` + `plotly` (no ngrok /
external server needed). Includes:
- **Woreda drill-down**: actual vs XGBoost vs GRU-implied trend
- **National map**: burden tier / forecast cases by woreda centroid
- **Supply chain view**: optimized AL coverage by region


In [ ]:
# Pre-compute a lookup of actual + XGBoost fitted values on the full (non-NaN) history,
# for the drill-down tab (fast to look up by woreda).
full_X = ml_df[FEATURES + CAT_FEATURES].copy()
for c in CAT_FEATURES:
    full_X[c] = full_X[c].astype("category")
ml_df["xgb_fitted"] = np.clip(xgb_model.predict(full_X), 0, None)

woreda_options = sorted(ml_df["ADM3_EN"].unique())

woreda_dd = widgets.Dropdown(options=woreda_options, description="Woreda:",
                              layout=widgets.Layout(width="400px"))
out1 = widgets.Output()

def plot_woreda(change=None):
    with out1:
        clear_output(wait=True)
        name = woreda_dd.value
        sub = ml_df[ml_df["ADM3_EN"] == name].sort_values("date")
        pcode = sub["ADM3_PCODE"].iloc[0]
        fc = forecast_df[forecast_df["ADM3_PCODE"] == pcode].sort_values("date")

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=sub["date"], y=sub["cases"], name="Actual cases", mode="lines"))
        fig.add_trace(go.Scatter(x=sub["date"], y=sub["xgb_fitted"], name="XGBoost (in-sample fit)",
                                  mode="lines", line=dict(dash="dot")))
        fig.add_trace(go.Scatter(x=fc["date"], y=fc["forecast_cases"], name="XGBoost forward forecast",
                                  mode="lines+markers", line=dict(color="firebrick")))
        fig.add_vrect(x0="2020-11-01", x1="2022-12-01", fillcolor="grey", opacity=0.1,
                      annotation_text="conflict window", annotation_position="top left")
        fig.update_layout(title=f"{name}: malaria cases, model fit & 6-month forecast",
                           xaxis_title="Date", yaxis_title="Cases", height=450)
        fig.show()

woreda_dd.observe(plot_woreda, names="value")
display(widgets.VBox([woreda_dd, out1]))
plot_woreda()


In [ ]:
# National map: latest-month forecast cases per woreda
latest_forecast = forecast_df[forecast_df["step_ahead"] == 1].merge(
    model_frame[["ADM3_PCODE", "shp_lat", "shp_lon", "ADM1_EN", "burden_tier"]].drop_duplicates("ADM3_PCODE"),
    on="ADM3_PCODE", how="left"
)

fig = px.scatter_mapbox(
    latest_forecast, lat="shp_lat", lon="shp_lon",
    size="forecast_cases", color="burden_tier",
    category_orders={"burden_tier": ["very high", "high", "moderate", "low"]},
    hover_name="ADM3_EN", hover_data=["ADM1_EN", "forecast_cases"],
    zoom=4.6, height=550, size_max=28,
    title="Next-month forecast cases by woreda (bubble size = forecast cases)"
)
fig.update_layout(mapbox_style="carto-positron", margin=dict(l=0, r=0, t=40, b=0))
fig.show()


In [ ]:
region_dd = widgets.Dropdown(options=["All regions"] + sorted(annual["ADM1_EN"].dropna().unique().tolist()),
                              description="Region:", layout=widgets.Layout(width="400px"))
out2 = widgets.Output()

def plot_supply(change=None):
    with out2:
        clear_output(wait=True)
        sub = annual if region_dd.value == "All regions" else annual[annual["ADM1_EN"] == region_dd.value]
        agg = sub.groupby("burden_tier").agg(
            need=("annual_need_al", "sum"), allocation=("al_allocation", "sum")
        ).reindex(["very high", "high", "moderate", "low"]).dropna(how="all").reset_index()

        fig = go.Figure()
        fig.add_bar(x=agg["burden_tier"], y=agg["need"], name="Annual need (AL courses)")
        fig.add_bar(x=agg["burden_tier"], y=agg["allocation"], name="Optimized allocation")
        fig.update_layout(barmode="group",
                           title=f"AL courses: need vs optimized allocation — {region_dd.value}",
                           height=420)
        fig.show()

        kpi = pd.DataFrame({
            "metric": ["Woredas", "Total need", "Total allocated", "Avg coverage %"],
            "value": [len(sub), f"{sub['annual_need_al'].sum():,.0f}",
                      f"{sub['al_allocation'].sum():,.0f}",
                      f"{sub['al_coverage_pct'].mean():.1f}%"]
        })
        display(kpi)

region_dd.observe(plot_supply, names="value")
display(widgets.VBox([region_dd, out2]))
plot_supply()


## 11. (Optional) Standalone Streamlit dashboard

If you'd rather have a shareable, standalone web dashboard instead of the
inline notebook widgets above, run the two cells below. They write a
`app.py` Streamlit app and launch it via a public tunnel URL so you (or
anyone you send the link to) can open it in a normal browser tab.


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(page_title="Ethiopia Malaria Dashboard", layout="wide")
st.title("Ethiopia Malaria Burden & Supply Chain Dashboard")

model_frame = pd.read_csv("model_frame.csv", parse_dates=["date"])
try:
    forecast_df = pd.read_csv("forecast_df.csv", parse_dates=["date"])
except FileNotFoundError:
    forecast_df = pd.DataFrame(columns=["ADM3_PCODE", "ADM3_EN", "date", "forecast_cases", "step_ahead"])
try:
    annual = pd.read_csv("annual_allocation.csv")
except FileNotFoundError:
    annual = pd.DataFrame()

tab1, tab2, tab3 = st.tabs(["Woreda drill-down", "National map", "Supply chain"])

with tab1:
    woreda = st.selectbox("Woreda", sorted(model_frame["ADM3_EN"].dropna().unique()))
    sub = model_frame[model_frame["ADM3_EN"] == woreda].sort_values("date")
    pcode = sub["ADM3_PCODE"].iloc[0]
    fc = forecast_df[forecast_df["ADM3_PCODE"] == pcode].sort_values("date")

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub["date"], y=sub["cases"], name="Actual cases"))
    if not fc.empty:
        fig.add_trace(go.Scatter(x=fc["date"], y=fc["forecast_cases"], name="Forecast",
                                  line=dict(color="firebrick")))
    fig.update_layout(title=f"{woreda}: malaria cases", height=450)
    st.plotly_chart(fig, use_container_width=True)

with tab2:
    latest = forecast_df[forecast_df["step_ahead"] == 1].merge(
        model_frame[["ADM3_PCODE", "shp_lat", "shp_lon", "ADM1_EN", "burden_tier"]].drop_duplicates("ADM3_PCODE"),
        on="ADM3_PCODE", how="left")
    if not latest.empty:
        fig = px.scatter_mapbox(latest, lat="shp_lat", lon="shp_lon", size="forecast_cases",
                                 color="burden_tier", hover_name="ADM3_EN", zoom=4.6, height=550)
        fig.update_layout(mapbox_style="carto-positron", margin=dict(l=0, r=0, t=0, b=0))
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Run the forecast step in the notebook and export forecast_df.csv to populate this map.")

with tab3:
    if not annual.empty:
        region = st.selectbox("Region", ["All regions"] + sorted(annual["ADM1_EN"].dropna().unique().tolist()))
        sub = annual if region == "All regions" else annual[annual["ADM1_EN"] == region]
        agg = sub.groupby("burden_tier").agg(need=("annual_need_al", "sum"),
                                              allocation=("al_allocation", "sum")).reset_index()
        fig = go.Figure()
        fig.add_bar(x=agg["burden_tier"], y=agg["need"], name="Need")
        fig.add_bar(x=agg["burden_tier"], y=agg["allocation"], name="Allocated")
        fig.update_layout(barmode="group", title=f"AL courses — {region}")
        st.plotly_chart(fig, use_container_width=True)
        c1, c2, c3 = st.columns(3)
        c1.metric("Woredas", len(sub))
        c2.metric("Total need", f"{sub['annual_need_al'].sum():,.0f}")
        c3.metric("Avg coverage", f"{sub['al_coverage_pct'].mean():.1f}%")
    else:
        st.info("Run the supply-chain optimization step in the notebook and export annual_allocation.csv.")


In [ ]:
# Export the tables the Streamlit app reads, then launch it behind a public tunnel.
forecast_df.to_csv("forecast_df.csv", index=False)
annual.to_csv("annual_allocation.csv", index=False)

!pip -q install streamlit
!npm -q install -g localtunnel 2>/dev/null

import subprocess, time
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)
!curl -s ipv4.icanhazip.com   # this is the tunnel password if prompted
!npx localtunnel --port 8501


Click the printed `https://*.loca.lt` URL; if prompted for a "Tunnel
Password", paste the IP address printed just above it. Stop the app by
interrupting/restarting the Colab runtime.


## 12. Suggested next steps

- **Model**: try `reg:quantileerror` (XGBoost >= 2.0) for prediction intervals
  instead of point forecasts; add a static per-woreda embedding to the GRU;
  try a hybrid (GRU residuals feeding into XGBoost).
- **Leakage check**: re-run Section 5 with `burden_tier` removed to see the
  "harder but more honest" forecasting performance.
- **Optimization**: extend the LP to a multi-period rolling plan (allocate
  monthly, not annually), add a hard logistics-capacity constraint (e.g. total
  truck-hours available per month from `time_hr`), or optimize artesunate
  and AL jointly with a shared transport budget.
- **Validation**: back-test the forward forecast against held-out months you
  already have (walk-forward cross-validation) before trusting it for real
  planning decisions.
